In [2]:
#%pip install lightgbm xgboost catboost


In [3]:
import pandas as pd
from src.config import *
from src.utils import get_latest_file
from src.predictions import *

from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
pd.set_option("display.max_columns", None)



In [4]:

# Paramètres
home_team_name = "Oklahoma City Thunder"
away_team_name = "Indiana Pacers"

home_odds = 1.22  # Cote de l'équipe à domicile
away_odds = 4.50  # Cote de l'équipe à l'extérieur

match_date = pd.to_datetime("2025-06-05")  # Date du match

# Chargement du dataset complet
dataset_path = get_latest_file(DATA_FINAL_DATASET_DIR)
full_df = pd.read_csv(dataset_path)
full_df["GAME_DATE"] = pd.to_datetime(full_df["GAME_DATE"])

# Mapping des noms d'équipes vers leurs IDs
team_mapping_file = get_latest_file(DATA_TEAMS_DIR)  # Fichier contenant les correspondances noms <-> IDs

team_mapping = pd.read_csv(team_mapping_file)

print("Mapping des équipes:")
print(team_mapping.head())


home_team_id = team_mapping.loc[team_mapping["full_name"] == home_team_name, "id"].values[0]
away_team_id = team_mapping.loc[team_mapping["full_name"] == away_team_name, "id"].values[0]






Mapping des équipes:
           id             full_name abbreviation         city          state  \
0  1610612737         Atlanta Hawks          ATL      Atlanta        Georgia   
1  1610612738        Boston Celtics          BOS       Boston  Massachusetts   
2  1610612739   Cleveland Cavaliers          CLE    Cleveland           Ohio   
3  1610612740  New Orleans Pelicans          NOP  New Orleans      Louisiana   
4  1610612741         Chicago Bulls          CHI      Chicago       Illinois   

   year_founded  
0          1949  
1          1946  
2          1970  
3          2002  
4          1966  


In [5]:
# Filtrage du dataset pour inclure uniquement les données avant la date du match
cut_df = full_df[full_df["GAME_DATE"] < match_date].copy()

# Génération des features pour la prédiction
prediction_df = build_prediction_rows(home_team_id=home_team_id, away_team_id=away_team_id, home_odds=home_odds, away_odds=away_odds, dataset=cut_df, match_date=match_date)



prediction_df

,IS_HOME,ROLL_fieldGoalsMade_traditional_3,ROLL_fieldGoalsMade_traditional_5,ROLL_fieldGoalsMade_traditional_10,ROLL_fieldGoalsMade_traditional_25,ROLL_fieldGoalsMade_traditional_50,ROLL_fieldGoalsMade_traditional_100,ROLL_fieldGoalsMade_traditional_200,ROLL_fieldGoalsAttempted_traditional_3,ROLL_fieldGoalsAttempted_traditional_5,ROLL_fieldGoalsAttempted_traditional_10,ROLL_fieldGoalsAttempted_traditional_25,ROLL_fieldGoalsAttempted_traditional_50,ROLL_fieldGoalsAttempted_traditional_100,ROLL_fieldGoalsAttempted_traditional_200,ROLL_threePointersMade_traditional_3,ROLL_threePointersMade_traditional_5,ROLL_threePointersMade_traditional_10,ROLL_threePointersMade_traditional_25,ROLL_threePointersMade_traditional_50,ROLL_threePointersMade_traditional_100,ROLL_threePointersMade_traditional_200,ROLL_threePointersAttempted_traditional_3,ROLL_threePointersAttempted_traditional_5,ROLL_threePointersAttempted_traditional_10,ROLL_threePointersAttempted_traditional_25,ROLL_threePointersAttempted_traditional_50,ROLL_threePointersAttempted_traditional_100,ROLL_threePointersAttempted_traditional_200,ROLL_freeThrowsMade_traditional_3,ROLL_freeThrowsMade_traditional_5,ROLL_freeThrowsMade_traditional_10,ROLL_freeThrowsMade_traditional_25,ROLL_freeThrowsMade_traditional_50,ROLL_freeThrowsMade_traditional_100,ROLL_freeThrowsMade_traditional_200,ROLL_freeThrowsAttempted_traditional_3,ROLL_freeThrowsAttempted_traditional_5,ROLL_freeThrowsAttempted_traditional_10,ROLL_freeThrowsAttempted_traditional_25,ROLL_freeThrowsAttempted_traditional_50,ROLL_freeThrowsAttempted_traditional_100,ROLL_freeThrowsAttempted_traditional_200,ROLL_reboundsOffensive_traditional_3,ROLL_reboundsOffensive_traditional_5,ROLL_reboundsOffensive_traditional_10,ROLL_reboundsOffensive_traditional_25,ROLL_reboundsOffensive_traditional_50,ROLL_reboundsOffensive_traditional_100,ROLL_reboundsOffensive_traditional_200,ROLL_reboundsDefensive_traditional_3,ROLL_reboundsDefensive_traditional_5,ROLL_reboundsDefensive_traditional_10,ROLL_reboundsDefensive_traditional_25,ROLL_reboundsDefensive_traditional_50,ROLL_reboundsDefensive_traditional_100,ROLL_reboundsDefensive_traditional_200,ROLL_reboundsTotal_traditional_3,ROLL_reboundsTotal_traditional_5,ROLL_reboundsTotal_traditional_10,ROLL_reboundsTotal_traditional_25,ROLL_reboundsTotal_traditional_50,ROLL_reboundsTotal_traditional_100,ROLL_reboundsTotal_traditional_200,ROLL_assists_traditional_3,ROLL_assists_traditional_5,ROLL_assists_traditional_10,ROLL_assists_traditional_25,ROLL_assists_traditional_50,ROLL_assists_traditional_100,ROLL_assists_traditional_200,ROLL_steals_traditional_3,ROLL_steals_traditional_5,ROLL_steals_traditional_10,ROLL_steals_traditional_25,ROLL_steals_traditional_50,ROLL_steals_traditional_100,ROLL_steals_traditional_200,ROLL_blocks_traditional_3,ROLL_blocks_traditional_5,ROLL_blocks_traditional_10,ROLL_blocks_traditional_25,ROLL_blocks_traditional_50,ROLL_blocks_traditional_100,ROLL_blocks_traditional_200,ROLL_turnovers_traditional_3,ROLL_turnovers_traditional_5,ROLL_turnovers_traditional_10,ROLL_turnovers_traditional_25,ROLL_turnovers_traditional_50,ROLL_turnovers_traditional_100,ROLL_turnovers_traditional_200,ROLL_foulsPersonal_traditional_3,ROLL_foulsPersonal_traditional_5,ROLL_foulsPersonal_traditional_10,ROLL_foulsPersonal_traditional_25,ROLL_foulsPersonal_traditional_50,ROLL_foulsPersonal_traditional_100,ROLL_foulsPersonal_traditional_200,ROLL_points_traditional_3,ROLL_points_traditional_5,ROLL_points_traditional_10,ROLL_points_traditional_25,ROLL_points_traditional_50,ROLL_points_traditional_100,ROLL_points_traditional_200,ROLL_MINUTES_PLAYED_3,ROLL_MINUTES_PLAYED_5,ROLL_MINUTES_PLAYED_10,ROLL_MINUTES_PLAYED_25,ROLL_MINUTES_PLAYED_50,ROLL_MINUTES_PLAYED_100,ROLL_MINUTES_PLAYED_200,ROLL_pointsOffTurnovers_misc_3,ROLL_pointsOffTurnovers_misc_5,ROLL_pointsOffTurnovers_misc_10,ROLL_pointsOffTurnovers_misc_25,ROLL_pointsOffTurnovers_misc_50,ROLL_pointsOffTurnovers_misc_100,ROLL_pointsOffTurnovers_misc_200,ROLL_poi

In [6]:
import joblib

#open last stacking model from DATA_MODELS_DIR 
stacking_model_path = get_latest_file(DATA_MODELS_DIR)
print("Loaded stacking model:", stacking_model_path)

# Charger le modèle de stacking
pipeline = joblib.load(stacking_model_path)


# Colonnes à ignorer
#drop_cols = COLS_TO_DROP_TARGET_IS_WIN + COLS_ODDS
X_pred = prediction_df.copy()

# ⚠️ Réordonner les colonnes si besoin
#X_pred = X_pred[[col for col in pipeline.named_steps['scaler'].get_feature_names_out() if col in X_pred.columns]]

# Prédictions
pred_classes = pipeline.predict(X_pred)
pred_probas = pipeline.predict_proba(X_pred)[:, 1]

# Résultats
results_df = prediction_df[['TEAM_ID', 'IS_HOME']].copy()
results_df['PREDICTED_WIN'] = pred_classes
results_df['WIN_PROBA'] = pred_probas

# Affichage lisible avec nom d’équipe
results_df = results_df.merge(team_mapping[['id', 'full_name']], left_on='TEAM_ID', right_on='id', how='left')
results_df = results_df[['full_name', 'IS_HOME', 'PREDICTED_WIN', 'WIN_PROBA']].rename(columns={'full_name': 'TEAM'})

display(results_df)


# Affichage des résultats
for idx, team_id in enumerate(prediction_df["TEAM_ID"]):
    team_name = team_mapping.loc[team_mapping["id"] == team_id, "full_name"].values[0]
    print(f"{team_name}: {pred_probas[idx]*100:.2f}% de chances de gagner")


Loaded stacking model: data\models\stacking_model_target_pointdiff_2025-06-05_03-26-42.joblib


e:\Documents_\Dev\NBA_Predictor\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


AttributeError: This 'Pipeline' has no attribute 'predict_proba'